# Chapter 8: System Design and Engineering Judgment

Estimated time: ~5 hours.

Prerequisites: Chapters 1-7. This chapter is a synthesis, not a new technique. Every
question in its framework points back at a chapter you've already built real code for.

## Concept: a repeatable framework, not a list of things to remember

A system-design interview question ("design an agent that does X") is rarely testing whether
you know individual techniques; Chapters 1 through 7 already covered those. It's testing
whether you can walk through a design in some repeatable order, out loud, without skipping
straight to "I'd use a ReAct loop with these five tools" before establishing whether an agent
is even the right tool for the job. This chapter's framework is nine questions, in order,
each one mapping directly onto a chapter you've already built:

| # | Question | Chapter it draws on |
|---|---|---|
| 0 | Does this actually need an agent? | (this chapter, next section) |
| 1 | What does success look like, and what does a failure cost? | — |
| 2 | What control flow fits — single call, fixed workflow, ReAct agent, or a multi-agent team? | Ch 1-2 |
| 3 | Does it need retrieval/grounding, and against what corpus? | Ch 3 |
| 4 | What's the reliability story — caching, retries, circuit breakers? | Ch 4 |
| 5 | What's the cost/latency budget, and does model routing help? | Ch 5 |
| 6 | What's the security surface — untrusted content, tool privilege? | Ch 6 |
| 7 | What tools does it need, and what happens when one fails? | Ch 7 |
| 8 | How will you know it's working, in production, over time? | Ch 3 (offline), Ch 9 (online) |

The order matters. Question 0 comes first on purpose: jumping straight to "here's my
architecture" without first asking whether an agent is the right call at all is one of the
most common tells that someone hasn't actually shipped one. Questions 1 and 2 come before
anything about tools or retrieval, because the failure cost and the control-flow choice
constrain everything that follows. A workflow that fails safely doesn't need the same
reliability investment as an autonomous agent making irreversible calls.

## When not to use an agent at all

This deserves its own section rather than being buried as "question 0" in a table, because
it's the single most common gap in an otherwise-competent design answer. An LLM agent is the
right tool when a task genuinely needs judgment under uncertainty across multiple steps:
deciding what to do next based on what just happened, not just executing a fixed sequence.
Most of the time someone reaches for an agent anyway, it's for one of five reasons, and none
of them hold up.

The most common is that the task is actually deterministic. If the "decision" the agent
would make is really just `if/elif/else` over well-understood cases, plain code is faster,
cheaper, and doesn't need Chapter 6's entire threat model. An agent adds nondeterminism and a
security surface to a problem that didn't have either. Close behind it is reaching for an
agent when a single LLM call, no loop, no tools, already solves it: summarization,
classification, extraction, and rewriting tasks are often genuinely single-shot. Chapter 1's
distinction between a chatbot, a workflow, and an agent exists precisely so this isn't a
judgment call made from vibes (see `curriculum/01_fundamentals.ipynb`'s opening comparison).
If there's no multi-step decision to make, there's no agent to build.

Two more signs matter enough to name on their own. If the failure cost is high and the task
is high-volume enough that a small error rate is unacceptable at scale, and no verification
step can catch it before it causes harm, an agent's probabilistic nature (Chapter 1) is a bad
fit for irreversible, high-volume, low-tolerance actions unless it's paired with a hard gate
(Chapter 6's least-privilege and policy-check patterns) that doesn't depend on the model
getting it right every time. And if latency requirements are tighter than an LLM round-trip,
let alone a multi-step agent loop, can meet, Chapter 5's latency decomposition applies
directly: if the budget is tens of milliseconds, no amount of model routing closes that gap.

The last one is the easiest to miss because it isn't about the task at all: you can't define
what "success" looks like well enough to evaluate it. If there's no way to tell whether the
agent did the right thing (Chapter 3's evaluation metrics, or even a simpler pass/fail
check), you can't safely iterate on it in production either. That's a design smell
independent of whether an agent is technically capable of the task.

None of this means "don't build agents." It means the honest answer to "would you use an
agent here" is sometimes "no, and here's the simpler thing I'd build instead," and being able
to say that convincingly is itself part of what this chapter's interview drill checks.

## The framework as a checklist

This is a small, real artifact, not just a table to memorize: a `DesignDoc` you can actually
fill in against any prompt, including the studios below and the mock interview in
`interview_prep/`.

In [1]:
from dataclasses import dataclass, field, fields


@dataclass
class DesignDoc:
    scenario: str
    needs_an_agent: str = ""          # question 0
    success_and_failure_cost: str = ""  # question 1
    control_flow: str = ""            # question 2 (Ch 1-2)
    retrieval_and_grounding: str = ""  # question 3 (Ch 3)
    reliability_story: str = ""       # question 4 (Ch 4)
    cost_latency_budget: str = ""     # question 5 (Ch 5)
    security_surface: str = ""        # question 6 (Ch 6)
    tools_and_failure_modes: str = ""  # question 7 (Ch 7)
    evaluation_plan: str = ""         # question 8 (Ch 3, Ch 9)

    def blank_fields(self) -> list[str]:
        return [f.name for f in fields(self) if f.name != "scenario" and not getattr(self, f.name)]


doc = DesignDoc(scenario="(fill this in per studio below)")
print(f"Framework fields to complete: {doc.blank_fields()}")


Framework fields to complete: ['needs_an_agent', 'success_and_failure_cost', 'control_flow', 'retrieval_and_grounding', 'reliability_story', 'cost_latency_budget', 'security_surface', 'tools_and_failure_modes', 'evaluation_plan']


## Design-doc studios

Three scenarios. For each, work through all nine questions from the framework above on your
own, in this notebook, on paper, out loud, however you actually rehearse, before checking
`solutions/ch08_system_design_judgment_answers.md`. The studios are deliberately blank here;
fully worked versions live only in the solutions file, exactly like every other chapter's
interview-drill answers.

### Studio 1: a support-ticket triage agent

A company wants an agent that reads incoming support tickets, answers simple questions from
their help-center docs, and can issue refunds up to a policy-defined limit without a human in
the loop. (This is deliberately close to Chapter 6's mock system: designing the *system*
around it, this time, not attacking it.)

Work through all nine questions:

0. Does this actually need an agent?
1. What does success look like, and what does a failure cost?
2. What control flow fits?
3. Does it need retrieval/grounding, and against what corpus?
4. What's the reliability story?
5. What's the cost/latency budget?
6. What's the security surface?
7. What tools does it need, and what happens when one fails?
8. How will you know it's working, in production, over time?

### Studio 2: a codebase-aware pull-request review assistant

An engineering team wants a tool that, given a pull request's diff, drafts review comments
(flagging likely bugs, style inconsistencies, and missing tests) for a human reviewer to
accept, edit, or dismiss before anything is posted.

Work through all nine questions (same list as Studio 1).

### Studio 3: a research assistant over a large internal document set

A legal team wants a tool that answers questions about a large corpus of internal contracts
and policy documents, citing which document(s) each answer came from, for a team that will
lose trust in the tool immediately if it ever states something confidently that isn't
actually in the source documents.

Work through all nine questions (same list as Studio 1).

## Interview drill

Answer each of these on your own before checking
`solutions/ch08_system_design_judgment_answers.md`. Nothing here is answered inline in this
notebook.

1. Judgment call. A stakeholder asks you to "just add an agent" to a feature that currently
runs as a fixed, deterministic pipeline with no LLM involved at all. How do you respond, and
what would actually change your answer to "yes, an agent helps here"?

2. Design judgment. For Studio 1 (support-ticket triage) above, a stakeholder pushes back:
"why do we need retries, circuit breakers, AND a policy check on refunds — isn't that
overkill for one feature?" How do you justify multiple layers without it sounding like
you're just piling on techniques you know?

3. Cold scenario. You're asked, cold, in an interview: "design an agent that monitors a
company's cloud infrastructure and can restart failed services automatically." Walk through
your first three moves using this chapter's framework — what do you ask/establish before
proposing any architecture at all?

4. Meta-judgment. What's the difference between an interview answer that *lists* techniques
from Chapters 1-7 and one that actually demonstrates system-design judgment? What
specifically would you listen for as the interviewer?

## Recap

This chapter didn't introduce a new technique. It introduced an order to ask questions in,
tying every prior chapter's technique to the specific design question it answers: does this
need an agent at all; what control flow fits; does it need retrieval; what's the reliability
story; what's the cost/latency budget; what's the security surface; what tools does it need
and how do their failures get handled; and how will you know it's working. The "when not to
use an agent" section is worth carrying forward on its own. It's the single most common gap
between a technically-correct answer and a genuinely convincing one.

Next: Chapter 9 closes the loop from "designed" to "running in production": canary releases,
shadow deployment, prompt versioning, drift detection, rollback, and actually containerizing
this course's agent code.